# Práctica 4: Filtro de Partículas en 2D (Monte Carlo Localization)

En esta práctica implementaremos un **Filtro de Partículas en 2D** completo para localizar a un robot móvil en un entorno abierto continuo que posee 4 balizas fijas (landmarks).

### Escenario:
- El robot se mueve en una arena de $100 \times 100$ metros.
- Hay 4 balizas en las esquinas: $(20, 20)$, $(80, 20)$, $(20, 80)$ y $(80, 80)$.
- El robot posee un sensor de distancias que mide el rango a cada una de las 4 balizas ruidosamente.
- El robot inicia sin saber dónde está (Localización Global).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Estilo de visualización premium
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 10)
plt.rcParams['font.size'] = 11

---
## 1. El Simulador Físico del Robot

Programaremos una clase `Robot` que modele su estado cinemático $(x, y, \theta)$ y simule el movimiento ruidoso y las lecturas analógicas del sensor de rango a balizas.

In [ ]:
class Robot:
    def __init__(self, x=None, y=None, theta=None):
        # Si no se proveen posiciones, colocar en el centro de forma aleatoria
        self.x = x if x is not None else np.random.rand() * 100.0
        self.y = y if y is not None else np.random.rand() * 100.0
        self.theta = theta if theta is not None else np.random.rand() * 2 * np.pi
        
        # Ruido por defecto del robot real
        self.noise_forward = 0.5    # Varianza del motor lineal (m)
        self.noise_turn = 0.05      # Varianza del motor angular (rad)
        self.noise_sense = 2.0      # Varianza del sensor de distancia (m)
        
    def set_noise(self, f, t, s):
        self.noise_forward = f
        self.noise_turn = t
        self.noise_sense = s
        
    def move(self, turn, forward):
        # Aplicar comando con ruido físico real
        self.theta = (self.theta + turn + np.random.normal(0, self.noise_turn)) % (2 * np.pi)
        dist = forward + np.random.normal(0, self.noise_forward)
        self.x = self.x + dist * np.cos(self.theta)
        self.y = self.y + dist * np.sin(self.theta)
        
    def sense(self, landmarks):
        # Medir distancias a todas las balizas añadiendo ruido Gaussiano
        mediciones = []
        for lm in landmarks:
            dist = np.sqrt((self.x - lm[0])**2 + (self.y - lm[lm_index:=1])**2)
            mediciones.append(dist + np.random.normal(0, self.noise_sense))
        return np.array(mediciones)

---
## 2. Funciones del Filtro de Partículas

Implementaremos modularmente el ciclo de vida del filtro:

In [ ]:
def crear_particulas(M):
    """
    Genera M partículas de forma uniforme en la arena de [0, 100]
    """
    particulas = []
    for _ in range(M):
        p = Robot(x=np.random.rand()*100.0, y=np.random.rand()*100.0, theta=np.random.rand()*2*np.pi)
        particulas.append(p)
    return particulas

def predict_particles(particulas, turn, forward, noise_turn=0.1, noise_forward=1.0):
    """
    Mueve cada partícula aplicando el mismo control 'u' pero añadiendo ruido de proceso.
    """
    for p in particulas:
        p.set_noise(noise_forward, noise_turn, 0.0)
        p.move(turn, forward)

def update_particles_weights(particulas, z, landmarks, noise_sense=2.0):
    """
    Calcula los pesos de importancia de cada partícula evaluando la verosimilitud de la lectura z.
    """
    pesos = []
    for p in particulas:
        likelihood = 1.0
        for i, lm in enumerate(landmarks):
            # Distancia simulada que la partícula vería en su estado hipotético
            dist_sim = np.sqrt((p.x - lm[0])**2 + (p.y - lm[1])**2)
            # Medida real recibida por el robot real
            dist_real = z[i]
            
            # Evaluar en la Gaussiana de ruido del sensor (Likelihood)
            prob = norm.pdf(dist_real, loc=dist_sim, scale=noise_sense)
            likelihood *= prob
            
        pesos.append(likelihood)
        
    pesos = np.array(pesos)
    # Evitar divisiones por cero sumando un pequeño valor epsilon
    pesos += 1e-300
    # Normalización
    pesos /= np.sum(pesos)
    return pesos

def resample_systematic(particulas, pesos):
    """
    Implementa la Rueda de Remuestreo Sistemático (O(M)).
    """
    M = len(particulas)
    nuevas_particulas = []
    
    index = int(np.random.rand() * M)
    beta = 0.0
    max_w = np.max(pesos)
    
    for i in range(M):
        beta += np.random.rand() * 2.0 * max_w
        while beta > pesos[index]:
            beta -= pesos[index]
            index = (index + 1) % M
            
        # Duplicar partícula copiada
        copia = Robot(x=particulas[index].x, y=particulas[index].y, theta=particulas[index].theta)
        nuevas_particulas.append(copia)
        
    return nuevas_particulas

def estimar_estado(particulas, pesos):
    """
    Calcula la media ponderada del estado estimado por la nube de partículas.
    """
    x_est = 0.0
    y_est = 0.0
    sin_theta = 0.0
    cos_theta = 0.0
    
    for i, p in enumerate(particulas):
        x_est += p.x * pesos[i]
        y_est += p.y * pesos[i]
        sin_theta += np.sin(p.theta) * pesos[i]
        cos_theta += np.cos(p.theta) * pesos[i]
        
    theta_est = np.arctan2(sin_theta, cos_theta) % (2 * np.pi)
    return x_est, y_est, theta_est

---
## 3. Simulación y Visualización de la Nube de Partículas

In [ ]:
# Inicializaciones
np.random.seed(42)
landmarks = np.array([[20.0, 20.0], [80.0, 20.0], [20.0, 80.0], [80.0, 80.0]])
M = 1000  # 1000 partículas para una excelente densidad y visualización

# Crear robot real físicamente situado en (30, 30)
robot = Robot(x=30.0, y=30.0, theta=np.pi/4.0)
robot.set_noise(f=0.2, t=0.02, s=2.0)

# Inicializamos nube de partículas con localización global (dispersas en el mapa)
particulas = crear_particulas(M)

hist_real = []
hist_estimacion = []

# Definimos un recorrido de 12 pasos
movimientos = [
    (0.1, 5.0), (0.1, 5.0), (0.1, 5.0), (0.2, 6.0), (0.2, 6.0),
    (0.2, 6.0), (-0.1, 5.0), (-0.1, 5.0), (-0.2, 5.0), (-0.2, 5.0),
    (-0.1, 5.0), (-0.1, 5.0)
]

print("🎬 SIMULANDO MOVIMIENTO Y LOCALIZACIÓN MULTIMODAL...")

for paso, (turn, forward) in enumerate(movimientos, 1):
    # 1. Mover el robot real y medir ruidosamente
    robot.move(turn, forward)
    z = robot.sense(landmarks)
    
    # 2. FILTRO DE PARTÍCULAS
    # a) Predicción
    predict_particles(particulas, turn, forward, noise_turn=0.05, noise_forward=0.5)
    # b) Actualización de pesos
    pesos = update_particles_weights(particulas, z, landmarks, noise_sense=2.0)
    # c) Estimar posición antes de remuestrear
    x_est, y_est, _ = estimar_estado(particulas, pesos)
    
    # Registrar datos
    hist_real.append([robot.x, robot.y])
    hist_estimacion.append([x_est, y_est])
    
    # d) Remuestreo condicional (siempre en este caso didáctico)
    particulas = resample_systematic(particulas, pesos)
    
    # Graficar instantes clave: paso 1 (inicial), paso 3 (convergencia media), paso 12 (seguimiento)
    if paso in [1, 3, 12]:
        plt.figure(figsize=(8, 8))
        # Balizas
        plt.scatter(landmarks[:, 0], landmarks[:, 1], color='gold', marker='*', s=350, edgecolors='black', label='Balizas (Landmarks)')
        
        # Partículas
        coords_p = np.array([[p.x, p.y] for p in particulas])
        plt.scatter(coords_p[:, 0], coords_p[:, 1], color='#d62728', alpha=0.3, s=8, label=f'Nube de {M} Partículas')
        
        # Robot Real
        plt.scatter(robot.x, robot.y, color='green', marker='o', s=200, edgecolors='black', label='Robot Real')
        # Estimación
        plt.scatter(x_est, y_est, color='blue', marker='x', s=200, label='Posición Estimada')
        
        plt.xlim(0, 100)
        plt.ylim(0, 100)
        plt.title(f'Filtro de Partículas 2D - Paso {paso}', pad=15)
        plt.xlabel('Eje X (m)')
        plt.ylabel('Eje Y (m)')
        plt.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9)
        plt.show()
        
    print(f"⏱️ Paso {paso:02d} -> Posición Real: ({robot.x:.2f}, {robot.y:.2f}) | Estimación: ({x_est:.2f}, {y_est:.2f})")

hist_real = np.array(hist_real)
hist_estimacion = np.array(hist_estimacion)

# Gráfica de trayectoria final acumulada
plt.figure(figsize=(10, 10))
plt.scatter(landmarks[:, 0], landmarks[:, 1], color='gold', marker='*', s=300, edgecolors='black', label='Balizas')
plt.plot(hist_real[:, 0], hist_real[:, 1], 'g-', linewidth=3, label='Trayectoria Real')
plt.plot(hist_estimacion[:, 0], hist_estimacion[:, 1], 'b--', linewidth=2.5, label='Trayectoria Estimada (Partículas)')
plt.title('Trayectoria Acumulada Completa: Real vs Filtro de Partículas', pad=15)
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.legend(frameon=True, facecolor='white')
plt.show()